In [ ]:
# Installing required libraries
!pip install -q transformers datasets torch accelerate

# Downloading DistilBERT model and tokenizer to local cache
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
model_name = "distilbert-base-uncased"

# Downloading weights and saving locally (no API key required)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name, local_files_only=False)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2, local_files_only=False)

# Saving model and tokenizer for offline use
tokenizer.save_pretrained("./local_distilbert")
model.save_pretrained("./local_distilbert")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Loading required libraries
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset

In [ ]:
# Loading IMDb dataset from Hugging Face Datasets (public, no API key needed)
# Dataset link: https://huggingface.co/datasets/imdb
dataset = load_dataset("imdb")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Selecting smaller subset to reduce training time
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test  = dataset["test"].shuffle(seed=42).select(range(1000))

In [ ]:
# Loading DistilBERT tokenizer and model from local directory (offline)
tokenizer = DistilBertTokenizerFast.from_pretrained("./local_distilbert", local_files_only=True)
model = DistilBertForSequenceClassification.from_pretrained("./local_distilbert", num_labels=2, local_files_only=True)

In [ ]:
# Tokenizing IMDb text data for model input
max_length = 128

def tokenize_batch(example):
    return tokenizer(
        example["text"],
        truncation=True,                 # trim long reviews
        padding="max_length",            # pad short reviews
        max_length=max_length
    )

# Applying tokenizer to train and test sets
train_dataset = small_train.map(tokenize_batch, batched=True)
test_dataset  = small_test.map(tokenize_batch, batched=True)

# Removing text column and formatting for PyTorch
train_dataset = train_dataset.remove_columns(["text"]).with_format("torch")
test_dataset  = test_dataset.remove_columns(["text"]).with_format("torch")


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
# Setting training parameters
training_args = TrainingArguments(
    output_dir="./results",              # directory to save checkpoints
    eval_strategy="epoch",         # evaluate at end of each epoch
    per_device_train_batch_size=8,       # training batch size
    per_device_eval_batch_size=16,       # evaluation batch size
    num_train_epochs=1,                  # one epoch for speed
    learning_rate=2e-5,                  # learning rate
    logging_steps=100,                   # log interval
    report_to="none"                     # disables Hugging Face Hub upload
)

In [ ]:
# Creating Trainer object and starting training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer
)

trainer.train()

/tmp/ipython-input-2576015553.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,0.455600,0.391818


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=250, training_loss=0.48149945068359373, metrics={'train_runtime': 1807.1157, 'train_samples_per_second': 1.107, 'train_steps_per_second': 0.138, 'total_flos': 66233699328000.0, 'train_loss': 0.48149945068359373, 'epoch': 1.0})

In [ ]:
# Evaluating model on test data
metrics = trainer.evaluate(eval_dataset=test_dataset)
print(metrics)

{'eval_loss': 0.3918180763721466, 'eval_runtime': 215.9654, 'eval_samples_per_second': 4.63, 'eval_steps_per_second': 0.292, 'epoch': 1.0}


In [ ]:
# Multiple custom reviews testing
while True:
    text = input("Enter a movie review (or 'exit' to stop): ")
    if text.lower() == "exit":
        break
    print("Sentiment:", predict_sentiment(text))
    print()  # blank line for readability


Enter a movie review (or 'exit' to stop): "Tenet is a mind-bending masterpiece! Christopher Nolan once again proves his genius with an intelligent and thrilling plot. The action scenes are stunning and the sound design is incredible."
Sentiment: Positive 😊

Enter a movie review (or 'exit' to stop): "Tenet was visually stunning and exciting."
Sentiment: Positive 😊

Enter a movie review (or 'exit' to stop): "Tenet was confusing and too loud."
Sentiment: Negative 😞

Enter a movie review (or 'exit' to stop): "The concept was brilliant and well executed."
Sentiment: Positive 😊

Enter a movie review (or 'exit' to stop): "I couldn’t follow what was happening."
Sentiment: Negative 😞

Enter a movie review (or 'exit' to stop): i cant here dilog of side acotr only bgm sound is high
Sentiment: Negative 😞

Enter a movie review (or 'exit' to stop): "I didn’t understand the story, but the visuals blew my mind."
Sentiment: Negative 😞

Enter a movie review (or 'exit' to stop): exit
